In [1]:
from typing import Any, Dict, List

from autogen_agentchat.agents import AssistantAgent
from autogen_agentchat.conditions import HandoffTermination, TextMentionTermination
from autogen_agentchat.messages import HandoffMessage
from autogen_agentchat.teams import Swarm
from autogen_agentchat.ui import Console
from autogen_ext.models.openai import OpenAIChatCompletionClient

In [2]:
def refund_flight (flight_PNR:str) ->str:
    return f"Refunded Flight with PNR {flight_PNR}"

In [3]:
api_key=None
model_client = OpenAIChatCompletionClient(model='gpt-4o',api_key=api_key)

In [4]:
travel_agent = AssistantAgent(
    "travel_agent",
    model_client=model_client,
    handoffs=["flights_refunder", "user"],
    system_message="""You are a travel agent.
    The flights_refunder is in charge of refunding flights.
    If you need information from the user, you must first send your message, then you can handoff to the user.
    Use TERMINATE when the travel planning is complete.""",
)

In [5]:
travel_agent = AssistantAgent(
    "travel_agent",
    model_client=model_client,
    handoffs=["flights_refunder", "user"],
    system_message="""You are a travel agent.
    The flights_refunder is in charge of refunding flights.
    If you need information from the user, you must first send your message, then you can handoff to the user.
    Use TERMINATE when the travel planning is complete.""",
)

In [6]:
flights_refunder = AssistantAgent(
    "flights_refunder",
    model_client=model_client,
    handoffs=["travel_agent", "user"],
    tools=[refund_flight],
    system_message="""You are an agent specialized in refunding flights.
    You only need flight PNR numbers to refund a flight.
    You have the ability to refund a flight using the refund_flight tool.
    If you need information from the user, you must first send your message, then you can handoff to the user.
    When the transaction is complete, handoff to the travel agent to finalize.""",
)

In [7]:
termination = HandoffTermination(target="user") | TextMentionTermination("TERMINATE")
team = Swarm([travel_agent, flights_refunder], termination_condition=termination)

In [ ]:
# if (isinstance(message, HandoffMessage))

In [ ]:
task = ' I want to refund my flight'

async def run_team_stream() -> None:

    task_result = await Console(team.run_stream(task = task))

    last_message = task_result.messages[-1]


    while ( isinstance(last_message,HandoffMessage) and last_message.target == 'user'):

        user_ka_message = input("User : ")

        task_result = await Console(team.run_stream(task = HandoffMessage(source='user',target=last_message.source,content=user_ka_message)))

        last_message = task_result.messages[-1]

await run_team_stream()

---------- TextMessage (user) ----------
 I want to refund my flight
---------- ToolCallRequestEvent (travel_agent) ----------
[FunctionCall(id='call_xQ7BfPdxkWeYUoxJV07Xqt80', arguments='{}', name='transfer_to_user'), FunctionCall(id='call_BkCxtjb6jjNcOhaqUM9TNgoj', arguments='{}', name='transfer_to_flights_refunder')]
---------- ToolCallExecutionEvent (travel_agent) ----------
[FunctionExecutionResult(content='Transferred to user, adopting the role of user immediately.', name='transfer_to_user', call_id='call_xQ7BfPdxkWeYUoxJV07Xqt80', is_error=False), FunctionExecutionResult(content='Transferred to flights_refunder, adopting the role of flights_refunder immediately.', name='transfer_to_flights_refunder', call_id='call_BkCxtjb6jjNcOhaqUM9TNgoj', is_error=False)]
---------- HandoffMessage (travel_agent) ----------
Transferred to user, adopting the role of user immediately.


c:\Users\admin\AutoGen\myenv212\lib\site-packages\autogen_agentchat\agents\_assistant_agent.py:1243: UserWarning: Multiple handoffs detected. Only the first is executed: ['transfer_to_user', 'transfer_to_flights_refunder']. Disable parallel tool calls in the model client to avoid this warning.
  handoff_output = cls._check_and_handle_handoff(


---------- HandoffMessage (user) ----------

---------- TextMessage (travel_agent) ----------
I've transferred you to the flights_refunder. They will assist you with your flight refund. If you have any more questions or need further assistance, feel free to let me know!
---------- ToolCallRequestEvent (travel_agent) ----------
[FunctionCall(id='call_sMXkQWGE8wTfrdMst9cetH5V', arguments='{}', name='transfer_to_user')]
---------- ToolCallExecutionEvent (travel_agent) ----------
[FunctionExecutionResult(content='Transferred to user, adopting the role of user immediately.', name='transfer_to_user', call_id='call_sMXkQWGE8wTfrdMst9cetH5V', is_error=False)]
---------- HandoffMessage (travel_agent) ----------
Transferred to user, adopting the role of user immediately.
---------- HandoffMessage (user) ----------
PNR12345
---------- TextMessage (travel_agent) ----------
Thank you for providing your PNR number. I'll make sure this information is passed on for your flight refund process. If there